# Embedding Layer 테스트

In [3]:
from transformers import AutoTokenizer
from loguru import logger
from model import Encoder, Decoder, Sqe2Seq

import torch

ImportError: cannot import name 'Sqe2Seq' from 'model' (/root/RNN_Translator/model.py)

In [1]:
def get_model_from_checkpoint(checkpoint, device):
    embedding_dim = int(checkpoint["embedding_dim"])
    hidden_dim = int(checkpoint["hidden_dim"])
    kor_vocab_size = int(checkpoint["kor_vocab_size"])
    en_vocab_size = int(checkpoint["en_vocab_size"])
    
    encoder = Encoder(kor_vocab_size, embedding_dim, hidden_dim)
    decoder = Decoder(en_vocab_size, embedding_dim, hidden_dim)
    model = Seq2Seq(encoder, decoder, padding_id = int(checkpoint.get("pad_token_id", 0)))
    
    model.load_state_dict(checkpoint["seq2seq_state_dict"], strict = True)
    model.to(device)
    model.eval()
    return model

def load_checkpoint(path, device):
    checkpoint = torch.load(path, map_location = device)
    
    required = ["seq2seq_state_dict", "embedding_dim", "hidden_dim", "kor_vocab_size", "en_vocab_size"]
    missing = [k for k in required if k not in checkpoint]
    
    if missing:
        raise KeyError(f"Missiong keys in checkpoint: {missing}")
    return checkpoint

device = "cuda" if torch.cuda.is_available() else "mps"

kor_tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
en_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

model_checkpoint_path = "checkpoints/best.pt"
model = load_checkpoint(model_checkpoint_path, device = device)

logger.info(f"Loaded checkpoint from {model_checkpoint_path}")

model = get_model_from_checkpoint(model, device = device)

embedding_matrix = model.encoder.embedding.weight.detach()
token_id = kor_tokenizer.encode("고양이", add_special_tokens = False)[0]

vec = embedding_matrix[token_id]
sims = torch.cosine_similarity(vec.unsqueeze(0), embedding_matrix)
topk = torch.topk(sims, k = 10)
print(kor_tokenizer.convert_ids_to_tokens(topk.indices.tolist()))

NameError: name 'torch' is not defined